# 리포트 66 — 코히어런트 배열이득은 10log₁₀N 상한에 −0.11~+0.47 dB 로 붙는다

> ### 한 일
> **한 지점 λ/2 배열의 소자 수를 늘려 가며 Pd=0.5 요구 SNR 을 재고, 그 이득을 열잡음만 상대할 때의 코히어런트 상한과 맞대 봤다.**

### 결과
1. 9모드 × N 전체에서 측정 이득은 상한 10log₁₀N 대비 -0.11 ⟨outputs/report05_derived.json : rx_gain.excess_min_db⟩ ~ +0.47 dB ⟨outputs/report05_derived.json : rx_gain.excess_max_db⟩ 다.
2. 결합 잡음전력/σ² = 0.99993 ⟨outputs/detection_rx_sweep.json : modes.W1.combine_ratio⟩ 로 잡음 보존을 확인했다 — 그래서 10log₁₀N 은 이상적 상한이다.
3. 초과분의 출처는 ECA 잔차다 — 감시신호가 `surv = √N·echo + dpi + noise` 이고 `dpi` 가 N 에 무관하게 고정이라 √N 이 잡음과 잔차 양쪽 대비로 표적을 올린다.
4. 로지스틱 재적합에서도 같은 부호가 나온다 — -0.07 ⟨outputs/report05_derived.json : rx_gain.excess_fit_min_db⟩ ~ +0.49 dB ⟨outputs/report05_derived.json : rx_gain.excess_fit_max_db⟩ 이고, 최대 초과분은 SNR50 몬테카를로 표준편차의 11.0 σ ⟨outputs/report05_derived.json : rx_gain.excess_in_sigma⟩ 배다.
5. 기하·규약 게이트 12 ⟨outputs/verify_freespace.json : summary.n_ran⟩건이 전부 통과했다(실패 0 ⟨outputs/verify_freespace.json : summary.n_fail⟩건).

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 배열 | 한 지점 λ/2 ULA 소자 N 개, 조향벡터는 참 표적 방향 — `src/experiment_detection.py:181` |
| 상한 | 10log₁₀N 은 **열잡음만** 상대할 때의 코히어런트 배열이득이고, 소자 간 결합·교정오차·위치오차가 0 인 이상적 값이다 |
| 대조군 | 격자 보간 대신 Pd 곡선에 로지스틱을 다시 적합해 같은 양을 두 번 잰다 |
| 몬테카를로 | K = 6000 ⟨outputs/report05_derived.json : rx_gain.K⟩ 회에서 SNR50 의 표준편차를 내고 초과분을 그 배수로 읽는다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_detection.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_freespace.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/detection_rx_sweep.json`, `outputs/verify_freespace.json`, `outputs/report05_derived.json` |
| 소요 | 검출 스윕 선언 예산 1800 s ⟨outputs/report05_derived.json : runtime.declared_breakdown_s.experiment_detection⟩ · 게이트 0.34 s ⟨outputs/report05_derived.json : runtime.verify_freespace_s⟩ |

---

## 수신소자를 늘리면

N 은 한 지점 λ/2 ULA 소자 수다(`src/experiment_detection.py:181`). 조향벡터를 참 표적 방향에 맞추고, 결합 잡음전력/σ² = 0.99993 ⟨outputs/detection_rx_sweep.json : modes.W1.combine_ratio⟩ 로 잡음 보존을 확인했다 — 그래서 10log₁₀N 은 **열잡음만** 상대할 때의 코히어런트 배열이득이고, 소자 간 결합·교정오차·위치오차가 0 인 **이상적 상한**이다.

| N | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| 측정 이득 (WiFi) = SNR50(1)−SNR50(N) | +0.00 dB | +3.37 dB | +5.24 dB | +6.44 dB |
| 열잡음 기준선 10log₁₀N | +0.00 dB | +3.01 dB | +4.77 dB | +6.02 dB |
| 차 (WiFi) | +0.00 dB | +0.36 dB | +0.47 dB | +0.42 dB |

출처 ⟨outputs/detection_rx_sweep.json : modes.W1.curves.*.snr50⟩

9모드 × N 전체에서 측정 이득은 그 상한 대비 -0.11 ⟨outputs/report05_derived.json : rx_gain.excess_min_db⟩ ~ +0.47 dB ⟨outputs/report05_derived.json : rx_gain.excess_max_db⟩ 다. 감시신호가 `surv = √N·echo + dpi + noise` 이고 ECA 잔차 `dpi` 는 N 에 무관하게 고정이라(`src/experiment_detection.py:284`), √N 이 잡음과 잔차 양쪽 대비로 표적을 올린다 — x 축 SNR 은 잡음 기준 정의다(`:238`).

| 검사 | 값 |
|---|---|
| Pd 곡선에 로지스틱을 다시 적합해 잰 초과분 | -0.07 ⟨outputs/report05_derived.json : rx_gain.excess_fit_min_db⟩ ~ +0.49 dB ⟨outputs/report05_derived.json : rx_gain.excess_fit_max_db⟩ |
| SNR50 의 몬테카를로 표준편차 (K = 6000 ⟨outputs/report05_derived.json : rx_gain.K⟩) | 0.043 dB ⟨outputs/report05_derived.json : rx_gain.snr50_mc_sigma_db⟩ |
| 최대 초과분 / 그 표준편차 | 11.0 σ ⟨outputs/report05_derived.json : rx_gain.excess_in_sigma⟩ |

![report05_pf5_multirx](../outputs/figures/report05_pf5_multirx.png)

**그림 1.** 수신소자를 늘렸을 때 얻는 감도는 이상적 코히어런트 상한에 얼마나 붙는가?

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| `SIONNA2_DPI_AMP=0` 대조군으로 Rx 스윕을 다시 돌린다 | 위 초과분이 ECA 잔차 대비 이득임이 대조군으로 확정된다 | `src/experiment_detection.py:115` → 이 편의 초과분 표 |